In [1]:
# compare_results_only.py
# Read ONLY out/251110/results/*.csv, summarize by scenario × λ (no fallbacks).

import os, glob
import pandas as pd
import matplotlib.pyplot as plt

RESULT_DIR = "out/251110/results"   # <-- results-only
WARMUP = 6000                       # minutes to ignore at start (20% of 30000)
SAVE_PREFIX = "_results_only_"

def main():
    # Load only results/*.csv (skip previously saved summaries starting with "_")
    paths = sorted(
        p for p in glob.glob(os.path.join(RESULT_DIR, "*.csv"))
        if not os.path.basename(p).startswith("_")
    )
    if not paths:
        raise FileNotFoundError(f"No CSVs found in {RESULT_DIR}. "
                                f"Expected combined outputs like FIFO_l0.28_<scenario>.csv")

    # Read and union
    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        # Ensure columns exist (fill if missing)
        for col in ["scenario","method","l","run","timestamp","status","cycle_time","activity"]:
            if col not in df.columns:
                df[col] = None
        df["__file"] = os.path.basename(p)
        dfs.append(df)
    raw = pd.concat(dfs, ignore_index=True)

    # Coerce numeric
    for c in ["l","run","timestamp","cycle_time"]:
        if c in raw.columns:
            raw[c] = pd.to_numeric(raw[c], errors="coerce")

    # Sanity: show which file contains which scenario/λ (so you always know the source)
    file_map = (raw.groupby("__file")[["scenario","l"]]
                   .agg(lambda s: sorted(pd.Series(s.dropna().unique()).tolist()))
                   .reset_index())
    print("\nFile → scenarios/λ found:")
    print(file_map.to_string(index=False))

    # Trim warm-up
    ss = raw[raw["timestamp"] >= WARMUP].copy()

    # Keys (handles single- or multi-run)
    keys = ["method","scenario","l","run"]

    # Completed-only for cycle time
    comp = ss[ss["status"] == "COMPLETE"].copy()
    ct = (comp.groupby(keys)["cycle_time"]
               .mean()
               .rename("mean_cycle_time")
               .reset_index())

    # Throughput = completions after warm-up / (t_max - warmup)
    tmax = (raw.groupby(keys)["timestamp"].max()
                 .rename("t_max").reset_index())
    tmax["window"] = (tmax["t_max"] - WARMUP).clip(lower=1)

    numc = (comp.groupby(keys).size()
                 .rename("num_completes").reset_index())

    summary = (ct.merge(numc, on=keys, how="left")
                 .merge(tmax, on=keys, how="left"))
    summary["throughput_per_min"] = summary["num_completes"] / summary["window"]
    summary = summary.sort_values(["scenario","l","method","run"])

    # Save summary
    os.makedirs(RESULT_DIR, exist_ok=True)
    out_csv = os.path.join(RESULT_DIR, f"{SAVE_PREFIX}scenario_lambda_summary.csv")
    summary.to_csv(out_csv, index=False)
    print("\nSaved:", out_csv)

    # Quick preview
    print("\nPreview (first rows):")
    print(summary.head(20).to_string(index=False))

    # Bar charts per λ
    for metric, ylabel in [("mean_cycle_time","Mean cycle time (min)"),
                           ("throughput_per_min","Throughput (cases/min)")]:
        for lam, d in summary.groupby("l"):
            if d.empty or pd.isna(lam): 
                continue
            d = d.sort_values(["scenario","method"])
            x = range(len(d))
            plt.figure(figsize=(9,5))
            plt.bar(list(x), d[metric].values)
            labels = [f"{s}\n({m})" for s,m in zip(d["scenario"], d["method"])]
            plt.xticks(list(x), labels, rotation=45, ha="right")
            plt.title(f"{metric} by scenario (λ={lam}, results-only)")
            plt.ylabel(ylabel)
            plt.tight_layout()
            outp = os.path.join(RESULT_DIR, f"{SAVE_PREFIX}{metric}_lambda_{lam}.png")
            plt.savefig(outp, dpi=150)
            plt.close()
            print("Saved plot:", outp)

if __name__ == "__main__":
    main()



File → scenarios/λ found:
                                                     __file                                       scenario      l
            FIFO_l0.28_actuator_manufacturing_no_rework.csv             [actuator_manufacturing_no_rework] [0.28]
          FIFO_l0.28_actuator_manufacturing_with_rework.csv           [actuator_manufacturing_with_rework] [0.28]
  FIFO_l0.28_actuator_mfg_pooledM_dedicatedA1_no_rework.csv   [actuator_mfg_pooledM_dedicatedA1_no_rework] [0.28]
FIFO_l0.28_actuator_mfg_pooledM_dedicatedA1_with_rework.csv [actuator_mfg_pooledM_dedicatedA1_with_rework] [0.28]
            FIFO_l0.28_all_dedicated_sticky_with_rework.csv             [all_dedicated_sticky_with_rework] [0.28]

Saved: out/251110/results\_results_only_scenario_lambda_summary.csv

Preview (first rows):
method                                     scenario    l  run  mean_cycle_time  num_completes        t_max       window  throughput_per_min
  FIFO             actuator_manufacturing_no_rework 0.28 